# California Parks Virtual Assistant

### Final Project

Fall 2025

CMPE 259 - Natural Language Processing

Author: Paul Junver Soriano

# Description

smth...

# Project Set Up

In [1]:
# Get data and Python modules from repo
!git clone https://github.com/paulsoriiiano/cmpe-259-project
!mv cmpe-259-project/data .
!mv cmpe-259-project/src .
!rm -rf cmpe-259-project

Cloning into 'cmpe-259-project'...
remote: Enumerating objects: 156, done.
remote: Counting objects: 100% (156/156), done.
remote: Compressing objects: 100% (108/108), done.
remote: Total 156 (delta 73), reused 109 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (156/156), 580.19 KiB | 5.42 MiB/s, done.
Resolving deltas: 100% (73/73), done.


In [2]:
# Install required libraries

%%bash
pip install faiss-cpu jq
pip install langchain langchain-community langchain_core langchain-huggingface langchain-mistralai langchain-openai
pip install huggingface_hub transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.1/757.1 kB 43.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of langchain-community to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-huggingface to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-mistralai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of langchain-openai to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.7 M

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [3]:
from huggingface_hub import login
from google.colab import userdata

login(new_session=False)
hf_token = userdata.get("HF_TOKEN")         # Get HuggingFace API Token
ms_token = userdata.get("MISTRAL_TOKEN")    # Get MistralAI API Token

# Load models

In [4]:
from src.llm_utils import load_model

small_llm = load_model(ms_token, "small")
large_llm = load_model(hf_token, "large")

In [5]:
query = "What is the weather like this weekend in Antelope Valley?"

In [6]:
print(small_llm.invoke(query).content)

I'm unable to provide real-time weather updates. For the most accurate and up-to-date weather forecast, I recommend checking a reliable weather website or app for the specific dates and location in Antelope Valley, California.


In [7]:
print(large_llm.invoke(query).content)

I'm a large language model, I don't have have access to real-time weather information. But I can suggest some ways for you to find out the current weather forecast for Antelope Valley.

You can check online weather websites such as:

1. National Weather Service (NWS) - [www.weather.gov](http://www.weather.gov)
2. AccuWeather - [www.accuweather.com](http://www.accuweather.com)
3. Weather.com - [www.weather.com](http://www.weather.com)

You can also check mobile apps like Dark Sky or Weather Underground for hyperlocal weather forecasts.

Additionally, you can search for "Antelope Valley weather forecast" or "weather in Antelope Valley this weekend" to get the latest updates.

Please note that Antelope Valley is a large region in northern Los Angeles County, California, so the weather forecast may vary depending on the specific location within the valley.


# Get vector database

In [8]:
from src.vector_db import build_vector_db

retriever = build_vector_db().as_retriever()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Build RAG chains

In [9]:
from src.rag_pipeline import build_rag_chain
rag_chain_small = build_rag_chain(small_llm, retriever)
rag_chain_large = build_rag_chain(large_llm, retriever)

In [10]:
query = "Which state beaches allow dogs?"

print(f"Small LLM response: \n\n{rag_chain_small.invoke(query)}")
print("\n======================================================\n")
print(f"Large LLM response: \n\n{rag_chain_large.invoke(query)}")

/content/src/rag_pipeline.py:17: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rag_docs = retriever.get_relevant_documents(query)


Small LLM response: 

Dogs on leashes are allowed at Monterey State Beach (South of the Monterey Beach Resort hotel), Asilomar State Beach, Carmel River State Beach, and Garrapata State Park, but are not allowed at Salinas River State Beach. Some state beaches have additional restrictions to protect the threatened western snowy plover, such as Salmon Creek Beach and Bodega Dunes Beach where dogs, fires, and camping are strictly prohibited. These new policies and restrictions are being implemented by California State Parks.

Sources:
- Salinas River State Beach information from https://www.parks.ca.gov/?page_id=573
- Dog rules information from https://www.parks.ca.gov/?page_id=576
- Sonoma Coast State Park information from https://www.parks.ca.gov/?page_id=451 (implied from the context that Salmon Creek Beach and Bodega Dunes Beach are part of Sonoma Coast State Park)


Large LLM response: 

You haven't provided a question. Please provide a question related to California parks and outdo